# 08 Final test evaluation

One-time evaluation on the held-out test set (June–December 2020), after every design decision was fixed on validation. The plan below was written before running any cell.

In [1]:
import sys; sys.path.append("..")
import json, numpy as np, pandas as pd, joblib
from pathlib import Path
from sklearn.metrics import average_precision_score, precision_recall_curve
from src.features import add_features, FEATURES
from src.policy import decide_thresholds, decide_expected_cost, realized_cost

pd.set_option("display.width", 200)
model = joblib.load("../models/xgb_v1.joblib")
policy = json.load(open("../models/policy.json"))
C = policy["costs"]

raw_tr = pd.read_csv("../data/raw/fraudTrain.csv", index_col=0, parse_dates=["trans_date_trans_time"])
raw_te = pd.read_csv("../data/raw/fraudTest.csv", index_col=0, parse_dates=["trans_date_trans_time"])
raw_tr["source"], raw_te["source"] = "train", "test"

full = add_features(pd.concat([raw_tr, raw_te], ignore_index=True))
full["prev_is_fraud"] = full.groupby("cc_num")["is_fraud"].shift(fill_value=0)
test = full[full["source"] == "test"].copy()

t = test["trans_date_trans_time"]
print(f"Test: {len(test):,} rows | fraud {test['is_fraud'].mean():.3%} | {t.min()} → {t.max()}")

Test: 555,719 rows | fraud 0.386% | 2020-06-21 12:14:25 → 2020-12-31 23:59:34


In [2]:
test["score"] = model.predict_proba(test[FEATURES])[:, 1]
y, amt, p = test["is_fraud"].to_numpy(), test["amt"].to_numpy(), test["score"].to_numpy()
labels = np.where(y == 1, "fraud", "legit")

def precision_at_recall(y, scores, target):
    precision, recall, _ = precision_recall_curve(y, scores)
    return precision[recall >= target].max()

report = {"test_rows": int(len(test)), "test_fraud_rate": float(y.mean()),
          "pr_auc": float(average_precision_score(y, p))}
for r in [0.7, 0.8, 0.9]:
    report[f"precision_at_recall_{int(r * 100)}"] = float(precision_at_recall(y, p, r))
print(json.dumps(report, indent=2))

bins = [0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.9, 1.0]
test["bin"] = pd.cut(test["score"], bins, include_lowest=True)
print(test.groupby("bin", observed=True).agg(n=("is_fraud", "size"),
                                             mean_score=("score", "mean"),
                                             fraud_rate=("is_fraud", "mean")).round(4).to_string())

{
  "test_rows": 555719,
  "test_fraud_rate": 0.0038598644278853163,
  "pr_auc": 0.962159178939282,
  "precision_at_recall_70": 0.9954809554551324,
  "precision_at_recall_80": 0.988479262672811,
  "precision_at_recall_90": 0.922673031026253
}
                      n  mean_score  fraud_rate
bin                                            
(-0.001, 0.001]  542958      0.0000      0.0000
(0.001, 0.005]     6586      0.0022      0.0027
(0.005, 0.01]      1369      0.0070      0.0088
(0.01, 0.05]       1755      0.0223      0.0211
(0.05, 0.1]         395      0.0707      0.0835
(0.1, 0.25]         431      0.1562      0.1462
(0.25, 0.5]         226      0.3559      0.3407
(0.5, 0.9]          309      0.7241      0.7055
(0.9, 1.0]         1690      0.9894      0.9905


In [3]:
no_model = amt[y == 1].sum()
policies = {
    "tuned thresholds": decide_thresholds(p, policy["t_otp"], policy["t_block"]),
    "expected cost":    decide_expected_cost(p, amt, C),
}
for name, actions in policies.items():
    loss, friction = realized_cost(actions, y, amt, C)
    print(f"{name:16s} | fraud loss ${loss:>9,.0f} | friction ${friction:>7,.0f} | "
          f"total ${loss + friction:>9,.0f} | cost cut {1 - (loss + friction) / no_model:.2%}")
    print(pd.crosstab(labels, actions, rownames=["actual"], colnames=["action"]), "\n")

final = policies["expected cost"]
loss, friction = realized_cost(final, y, amt, C)
report.update({
    "no_model_cost": float(no_model),
    "policy_cost": loss + friction,
    "cost_cut": 1 - (loss + friction) / no_model,
    "fraud_recall_any_action": float((final[y == 1] != "approve").mean()),
    "genuine_friction_rate": float((final[y == 0] != "approve").mean()),
    "genuine_block_rate": float((final[y == 0] == "block").mean()),
})

tuned thresholds | fraud loss $   17,331 | friction $ 13,484 | total $   30,815 | cost cut 97.28%
action  approve  block   otp
actual                      
fraud        40   1969   136
legit    550416    256  2902 

expected cost    | fraud loss $   14,416 | friction $ 12,296 | total $   26,712 | cost cut 97.64%
action  approve  block   otp
actual                      
fraud        79   1852   214
legit    551360    281  1933 



In [4]:
# Thresholds each scenario got when tuned on validation (sensitivity table)
scenarios = {
    "base":             (C,                              0.0075, 0.25),
    "expensive blocks": ({**C, "block_friction": 100.0}, 0.0075, 0.50),
    "annoying OTP":     ({**C, "otp_friction": 5.0},     0.0100, 0.25),
    "weak OTP":         ({**C, "otp_stop_rate": 0.5},    0.0075, 0.10),
}
for name, (c, t_otp, t_block) in scenarios.items():
    cuts = {}
    for pname, acts in [("thresholds", decide_thresholds(p, t_otp, t_block)),
                        ("expected cost", decide_expected_cost(p, amt, c))]:
        l, f = realized_cost(acts, y, amt, c)
        cuts[pname] = f"{1 - (l + f) / no_model:.2%}"
    print(f"{name:16s} | " + " | ".join(f"{k}: {v}" for k, v in cuts.items()))

frauds = test[test["is_fraud"] == 1].assign(flagged=final[y == 1] != "approve")
spree = frauds.groupby("prev_is_fraud")["flagged"].agg(recall="mean", n="size")
spree.index = spree.index.map({0: "first fraud on card", 1: "fraud after a fraud"})
print(spree)
report["first_fraud_recall"] = float(spree.loc["first fraud on card", "recall"])

base             | thresholds: 97.28% | expected cost: 97.64%
expensive blocks | thresholds: 96.43% | expected cost: 96.68%
annoying OTP     | thresholds: 96.70% | expected cost: 97.27%
weak OTP         | thresholds: 95.93% | expected cost: 96.93%
                       recall     n
prev_is_fraud                      
first fraud on card  0.869159   214
fraud after a fraud  0.973589  1931


In [5]:
Path("../reports").mkdir(exist_ok=True)
with open("../reports/test_metrics.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

{
  "test_rows": 555719,
  "test_fraud_rate": 0.0038598644278853163,
  "pr_auc": 0.962159178939282,
  "precision_at_recall_70": 0.9954809554551324,
  "precision_at_recall_80": 0.988479262672811,
  "precision_at_recall_90": 0.922673031026253,
  "no_model_cost": 1133324.6800000002,
  "policy_cost": 26712.429999999997,
  "cost_cut": 0.9764300288598674,
  "fraud_recall_any_action": 0.9631701631701631,
  "genuine_friction_rate": 0.003999465292806382,
  "genuine_block_rate": 0.0005076105452929509,
  "first_fraud_recall": 0.8691588785046729
}


## Results

**Model quality**
- Test set: 555,719 transactions with 0.386% fraud, 37% rarer than validation (0.615%).
- PR-AUC 0.962 (validation: 0.976). Precision at 80% / 90% recall: 98.8% / 92.3%.
- Calibration held despite the lower fraud rate (predicted vs actual: 0.0223 vs 0.0211, 0.3559 vs 0.3407, 0.9894 vs 0.9905).

**Business outcome**
- Expected-cost rule: total cost $26,712, vs $30,815 for validation-tuned thresholds.
- 96.3% of frauds intercepted (OTP or block).
- 0.40% of genuine transactions saw any friction; 0.05% were blocked.
- Fraud-related cost cut by 97.6% ($1,133,325 → $26,712).
- The rule beat the thresholds under all four cost assumptions.

**Weakness**
- A card's first fraud is caught 86.9% of the time (214 cases), vs 97.4% for later frauds (1,931 cases).

All numbers are saved to `reports/test_metrics.json`. No model, feature, or policy changes were made after viewing these results.